# 14 ASG-RNN-like Benchmark

This notebook is a practical ASG-RNN-like adaptation for the existing M5 graph benchmark. Local project search did not find a verified ASG-RNN implementation, so the experiment below reuses the current protocol and builds a minimum viable graph-recurrent baseline.

Important: this is **ASG-RNN-like**, not an exact reproduction of the ASG-RNN paper.

## Existing Project Protocol

The benchmark settings are imported from `src/experiments/common_protocol.py` and `src/experiments/stat_benchmark_utils.py`.

- high-demand / stable: `FOODS_3_228_CA_1_validation`
- intermittent: `FOODS_2_044_CA_3_validation`
- low-volume: `HOBBIES_1_133_CA_4_validation`

Protocol: CA state filter, seed 42, latest 365 days, 28-day context, 28-day validation, 28-day test.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.common_protocol import OFFICIAL_BENCHMARK_PROTOCOL, protocol_summary_text
from src.experiments.stat_benchmark_utils import BENCHMARK_PRODUCTS

print(protocol_summary_text())
BENCHMARK_PRODUCTS

M5_GNN_SUBSET: max_days=365, context_length=28, val_days=28, test_days=28, seed=42, state_id=CA


[('FOODS_3_228_CA_1_validation', 'high_demand_stable'),
 ('FOODS_2_044_CA_3_validation', 'intermittent'),
 ('HOBBIES_1_133_CA_4_validation', 'low_volume')]

## ASG-RNN Feasibility Result

Local search found graph-recurrent components but no direct ASG-RNN implementation. The closest existing code is:

- `src/models/graph_forecast_model.py`: GRU encoder plus adjacency-based neighbor aggregation.
- `src/experiments/run_gnn_benchmark.py`: M5 product graph benchmark prototype.
- `src/experiments/run_correlation_matrix_16_products.py`: hybrid graph construction and graph artifacts.

This notebook uses a new runner, `src/experiments/run_asg_rnn_like_benchmark.py`, which adds price behavior and static node features to the graph-recurrent baseline.

## Model Design

The ASG-RNN-like baseline uses:

- dynamic inputs: scaled sales history and scaled sell-price history;
- static node features: M5 metadata and demand-profile features;
- graph: top-k hybrid product similarity from log-sales correlation, price correlation, metadata similarity, and demand-profile similarity;
- temporal encoder: GRU;
- graph aggregation: adjacency-weighted neighbor representation;
- target: one-step recursive point forecast over the final 28 test days.

In [2]:
from src.experiments.run_asg_rnn_like_benchmark import ASGRNNLikeConfig, DEFAULT_OUT_DIR, run_experiment

config = ASGRNNLikeConfig(
    epochs=18,
    expanded_products=48,
    top_k=5,
    seed=OFFICIAL_BENCHMARK_PROTOCOL.seed,
)

outputs = run_experiment(DEFAULT_OUT_DIR, config)
outputs

c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\.venv\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\.venv\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\.venv\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\.venv\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


{'out_dir': 'C:\\Users\\braya\\Documents\\Research\\aurex-demand-forecasting-main\\reports\\asg_rnn_like_benchmark',
 'metrics_path': 'C:\\Users\\braya\\Documents\\Research\\aurex-demand-forecasting-main\\reports\\asg_rnn_like_benchmark\\metrics.csv',
 'predictions_path': 'C:\\Users\\braya\\Documents\\Research\\aurex-demand-forecasting-main\\reports\\asg_rnn_like_benchmark\\predictions.csv',
 'training_summary_path': 'C:\\Users\\braya\\Documents\\Research\\aurex-demand-forecasting-main\\reports\\asg_rnn_like_benchmark\\training_summary.csv',
 'graph_summary_path': 'C:\\Users\\braya\\Documents\\Research\\aurex-demand-forecasting-main\\reports\\asg_rnn_like_benchmark\\graph_summary.csv'}

## Benchmark Metrics

Metrics are saved to `reports/asg_rnn_like_benchmark/metrics.csv`. The table below reports the official 16-product panel and the expanded 48-product panel.

In [3]:
import pandas as pd

metrics = pd.read_csv(DEFAULT_OUT_DIR / 'metrics.csv')
cols = [
    'run_name', 'benchmark_label', 'series_id', 'n_products',
    'mae', 'rmse', 'variance_ratio', 'flat_nonflat',
    'trend_correlation', 'shape_similarity', 'peak_detection_rate'
]
metrics[cols]

,run_name,benchmark_label,series_id,n_products,mae,rmse,variance_ratio,flat_nonflat,trend_correlation,shape_similarity,peak_detection_rate
0,official_16,intermittent,FOODS_2_044_CA_3_validation,16,1.142850,1.487611,0.033887,flat,0.382290,0.681553,0.0
1,official_16,high_demand_stable,FOODS_3_228_CA_1_validation,16,2.312238,2.818022,0.026132,flat,0.597300,0.637534,0.0
2,official_16,low_volume,HOBBIES_1_133_CA_4_validation,16,0.016742,0.016764,0.000000,flat,NaN,0.362815,1.0
3,expanded_48,high_demand_stable,FOODS_3_228_CA_1_validation,48,2.350785,2.892217,0.025735,flat,0.597234,0.674404,0.0
4,expanded_48,intermittent,FOODS_2_044_CA_3_validation,48,1.142131,1.706533,0.122850,non-flat,0.355906,0.625015,0.0
5,expanded_48,low_volume,HOBBIES_1_133_CA_4_validation,48,0.000000,0.000000,0.000000,flat,NaN,1.000000,1.0


## Expanded Product Panel

The expanded panel is selected reproducibly from CA M5 products. The rule forces the benchmark trio into the graph, then samples a balanced mix of stable/high, intermittent, and low-volume products using demand profile statistics from the latest 365 days.

In [4]:
expanded_meta = pd.read_csv(DEFAULT_OUT_DIR / 'graph_artifacts' / 'expanded_48_metadata.csv')
graph_summary = pd.read_csv(DEFAULT_OUT_DIR / 'graph_summary.csv')
display(expanded_meta['demand_class'].value_counts().rename_axis('demand_class').reset_index(name='count'))
display(graph_summary)

,demand_class,count
0,stable_or_high,16
1,intermittent,16
2,low_volume,16


,nodes,edges,density,connected_components,largest_component_size,run_name
0,16.0,51.0,0.425000,1.0,16.0,official_16
1,48.0,165.0,0.146277,1.0,48.0,expanded_48


## Saved Outputs

The runner saves:

- predictions: `reports/asg_rnn_like_benchmark/predictions.csv`
- metrics: `reports/asg_rnn_like_benchmark/metrics.csv`
- training summary: `reports/asg_rnn_like_benchmark/training_summary.csv`
- plots: `reports/asg_rnn_like_benchmark/figures/`
- graph artifacts: `reports/asg_rnn_like_benchmark/graph_artifacts/`
- graph summary: `reports/asg_rnn_like_benchmark/graph_summary.csv`

A concise note on self-learning and self-supervised learning is saved separately at `reports/self_learning_and_self_supervised_note.md`.